In [ ]:
# Полноценное обучение моделей модерации (текст + изображения)
#
# Ожидаемые parquet:
# - ../data/text_dataset.parquet: колонки минимум [text, label], опционально [id, split]
# - ../data/images_dataset.parquet: колонки [image_path] или [image_bytes], опционально [label, id, split]
#
# Артефакты сохраняются в ../artifacts/

from __future__ import annotations

import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# Важно: запускаем ноутбук из ML/notebooks, поэтому ../src доступен через sys.path
import sys
sys.path.append(str(Path('..').resolve() / 'src'))

from pinz_ml.paths import artifacts_dir, data_dir
from pinz_ml.data.parquet_loaders import (
    TextDatasetSpec,
    ImageDatasetSpec,
    load_text_dataset,
    load_image_dataset,
    random_split,
    split_by_column,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TEXT_PARQUET = data_dir() / 'text_dataset.parquet'
IMAGES_PARQUET = data_dir() / 'images_dataset.parquet'

print('TEXT_PARQUET:', TEXT_PARQUET)
print('IMAGES_PARQUET:', IMAGES_PARQUET)
print('ARTIFACTS:', artifacts_dir())

# Helpers

def set_torch_seed(seed: int = 42):
    import torch

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def classification_report_simple(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Мини-репорт без внешних либ (но sklearn всё равно будем использовать ниже для удобства)."""
    acc = float((y_true == y_pred).mean())
    return {'accuracy': acc}


set_torch_seed(SEED)
print('Готово: импорты/пути/seed')


In [ ]:
# 1) EDA + baseline для текста (TF-IDF + LogisticRegression)

import re
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')

assert TEXT_PARQUET.exists(), f"Не найден файл: {TEXT_PARQUET}. Положите parquet в ML/data/"

df_text = load_text_dataset(TextDatasetSpec(path=TEXT_PARQUET, text_col='text', label_col='label'))
print('Rows:', len(df_text))
print('Columns:', list(df_text.columns))
display(df_text.head())

# Базовая проверка качества
print('Missing text:', int(df_text['text'].isna().sum()))
print('Missing label:', int(df_text['label'].isna().sum()))
print('Duplicates:', int(df_text.duplicated(subset=['text', 'label']).sum()))

# Распределение классов
plt.figure(figsize=(6, 3))
df_text['label'].value_counts().sort_index().plot(kind='bar')
plt.title('Label distribution (text)')
plt.xlabel('label')
plt.ylabel('count')
plt.tight_layout()
plt.show()

# Длины (символы/токены)
df_text['len_chars'] = df_text['text'].astype(str).map(len)
df_text['len_tokens'] = df_text['text'].astype(str).str.split().map(len)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
sns.histplot(df_text['len_chars'], bins=60, ax=ax[0])
ax[0].set_title('Text length (chars)')
sns.histplot(df_text['len_tokens'], bins=60, ax=ax[1])
ax[1].set_title('Text length (tokens)')
plt.tight_layout()
plt.show()

# Небольшая нормализация текста для baseline
def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"https?://\S+", " URL ", s)
    s = re.sub(r"@\w+", " USER ", s)
    s = re.sub(r"#\w+", " HASHTAG ", s)
    s = re.sub(r"\d+", " NUM ", s)
    s = re.sub(r"[^a-zа-я0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_text['text_norm'] = df_text['text'].map(normalize_text)

# Разбиение (если есть split — используем, иначе random)
if 'split' in df_text.columns:
    train_df, val_df, test_df = split_by_column(df_text, split_col='split')
else:
    train_df, val_df, test_df = random_split(df_text, train_frac=0.8, val_frac=0.1, seed=SEED)

print('Split sizes:', len(train_df), len(val_df), len(test_df))

# TF-IDF + LogisticRegression baseline
baseline = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=150_000, min_df=2)),
        ('clf', LogisticRegression(max_iter=200, n_jobs=1, class_weight='balanced')),
    ]
)

baseline.fit(train_df['text_norm'], train_df['label'])
val_pred = baseline.predict(val_df['text_norm'])
print('Baseline (val)')
print(classification_report(val_df['label'], val_pred, digits=4))

cm = confusion_matrix(val_df['label'], val_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion matrix (val)')
plt.xlabel('pred')
plt.ylabel('true')
plt.tight_layout()
plt.show()

# Сохраняем baseline как артефакт (pickle)
import joblib
baseline_path = artifacts_dir() / 'text_baseline_tfidf_logreg.joblib'
joblib.dump(baseline, baseline_path)
print('Saved baseline to:', baseline_path)


In [ ]:
# 2) Fine-tuning Transformers для текста (большой цикл обучения + метрики)

import math
from typing import Dict

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = os.getenv('TEXT_MODEL_NAME', 'distilbert-base-uncased')
MAX_LEN = int(os.getenv('TEXT_MAX_LEN', '192'))

num_labels = int(df_text['label'].nunique())
print('num_labels:', num_labels)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HFTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]))
        return item

train_ds = HFTextDataset(train_df['text_norm'], train_df['label'], tokenizer, MAX_LEN)
val_ds = HFTextDataset(val_df['text_norm'], val_df['label'], tokenizer, MAX_LEN)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

def compute_metrics(eval_pred) -> Dict[str, float]:
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import accuracy_score, f1_score

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1_macro}

# Параметры обучения (достаточно "реальные", но не слишком тяжёлые)
out_dir = artifacts_dir() / 'text_transformer'
out_dir.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(out_dir),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    per_device_train_batch_size=int(os.getenv('TEXT_BATCH', '16')),
    per_device_eval_batch_size=int(os.getenv('TEXT_EVAL_BATCH', '32')),
    num_train_epochs=float(os.getenv('TEXT_EPOCHS', '2')),
    learning_rate=float(os.getenv('TEXT_LR', '2e-5')),
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Trainer готов. Для запуска обучения выполните: trainer.train()')
# trainer.train()

# После обучения можно сохранить:
# trainer.save_model(str(out_dir / 'best_model'))
# tokenizer.save_pretrained(str(out_dir / 'best_model'))


In [ ]:
# 3) EDA + обучение модели для изображений (ResNet18 fine-tune)

from io import BytesIO

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

assert IMAGES_PARQUET.exists(), f"Не найден файл: {IMAGES_PARQUET}. Положите parquet в ML/data/"

df_img = load_image_dataset(ImageDatasetSpec(path=IMAGES_PARQUET))
print('Rows:', len(df_img))
print('Columns:', list(df_img.columns))
display(df_img.head())

# Попробуем понять, как хранятся картинки
has_bytes = 'image_bytes' in df_img.columns
has_path = 'image_path' in df_img.columns
print('has_bytes:', has_bytes, 'has_path:', has_path)

# Статистика по label, если есть
if 'label' in df_img.columns:
    plt.figure(figsize=(6, 3))
    df_img['label'].value_counts().sort_index().plot(kind='bar')
    plt.title('Label distribution (images)')
    plt.tight_layout()
    plt.show()

# Визуализация нескольких картинок

def load_pil_from_row(row) -> Image.Image:
    if has_bytes and row.get('image_bytes') is not None:
        return Image.open(BytesIO(row['image_bytes'])).convert('RGB')
    if has_path and row.get('image_path'):
        return Image.open(row['image_path']).convert('RGB')
    raise ValueError('Row does not contain image_bytes/image_path')

sample_rows = df_img.sample(min(12, len(df_img)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
axes = axes.flatten()
for ax, (_, r) in zip(axes, sample_rows.iterrows()):
    try:
        im = load_pil_from_row(r)
        ax.imshow(im)
        ax.set_title(f"label={int(r.get('label', -1))}")
    except Exception as e:
        ax.text(0.1, 0.5, str(e))
    ax.axis('off')
plt.tight_layout()
plt.show()

# Разбиение
if 'split' in df_img.columns:
    train_img_df, val_img_df, test_img_df = split_by_column(df_img, split_col='split')
else:
    train_img_df, val_img_df, test_img_df = random_split(df_img, train_frac=0.8, val_frac=0.1, seed=SEED)

print('Image split sizes:', len(train_img_df), len(val_img_df), len(test_img_df))

# Dataset + transforms
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop((224, 224), scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ParquetImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, label_col: str = 'label'):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        im = load_pil_from_row(row)
        x = self.transform(im)
        y = int(row.get(self.label_col, 0))
        return x, y

train_ds = ParquetImageDataset(train_img_df, train_tf)
val_ds = ParquetImageDataset(val_img_df, eval_tf)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

train_loader = DataLoader(train_ds, batch_size=int(os.getenv('IMG_BATCH', '32')), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=int(os.getenv('IMG_EVAL_BATCH', '64')), shuffle=False, num_workers=0)

# Модель
num_img_labels = int(df_img['label'].nunique()) if 'label' in df_img.columns else 2
model_img = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_img.fc = nn.Linear(model_img.fc.in_features, num_img_labels)
model_img = model_img.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_img.parameters(), lr=float(os.getenv('IMG_LR', '3e-4')), weight_decay=0.01)

@torch.no_grad()
def eval_epoch(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    losses = []
    all_y = []
    all_p = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        losses.append(float(loss.item()))
        p = torch.argmax(logits, dim=-1)
        all_y.append(y.cpu().numpy())
        all_p.append(p.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    from sklearn.metrics import accuracy_score, f1_score

    return {
        'loss': float(np.mean(losses)) if losses else 0.0,
        'accuracy': float(accuracy_score(y_true, y_pred)) if len(y_true) else 0.0,
        'f1_macro': float(f1_score(y_true, y_pred, average='macro')) if len(y_true) else 0.0,
    }


def train_epochs(model: nn.Module, epochs: int = 3):
    best = -1.0
    best_path = artifacts_dir() / 'image_resnet18_best.pt'

    for epoch in range(1, epochs + 1):
        model.train()
        running = []
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running.append(float(loss.item()))

        val_metrics = eval_epoch(model, val_loader)
        train_loss = float(np.mean(running)) if running else 0.0
        print(f"epoch={epoch} train_loss={train_loss:.4f} val={val_metrics}")

        if val_metrics['f1_macro'] > best:
            best = val_metrics['f1_macro']
            torch.save(
                {
                    'model_state': model.state_dict(),
                    'num_labels': num_img_labels,
                },
                best_path,
            )
            print('saved best:', best_path)

    print('best f1_macro:', best)

print('Готово. Для запуска обучения выполните: train_epochs(model_img, epochs=3)')
# train_epochs(model_img, epochs=int(os.getenv('IMG_EPOCHS', '3')))
